In [27]:
import fitz 
import re

In [ ]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    pages_to_skip = {0, 1, 2, 3, 8, total_pages - 1}
    
    pages = []

    for page_num in range(total_pages):
        if page_num not in pages_to_skip:
            page = doc.load_page(page_num)
            
            raw_page_text = page.get_text().strip()
            
            lines = raw_page_text.split('\n')
            
            if len(lines) > 2:
                cleaned_text = '\n'.join(lines[:-2])
            else:
                cleaned_text = raw_page_text
                
            pages.append(cleaned_text)      
            
    return pages

pdf_path = "./three-days-of-happiness.pdf"
pages = extract_text_from_pdf(pdf_path)

raw_text = "\n\n".join(pages)

print(f"Text Extracted. Total len: {len(raw_text)} chars.")
print("--- Sample page ---")
print(pages[6])

Text Extracted. Total len: 294687 chars.
--- Sample page ---
Every class has some witty clown, and he was on the same train of
thought as me. “If I sold you the right to have my life, you wouldn’t even pay
three hundred yen, would you?” he said, to hearty laughs. I agreed with the
sentiment, but of course, he was only being sarcastically self-effacing for
laughs and attention. He clearly considered himself to be far more valuable to
the group than the boring, serious students—a fact I found detestable.
However, although the teacher told us there was no right answer, in fact,
there was. Ten years later, when I turned twenty, I actually did sell my future
life span and received something of value in return.
When I was a kid, I thought I would grow up to be someone important. I
believed I was exceedingly special compared with my peers. Unfortunately,
because my neighborhood was filled with extremely unimpressive parents
who gave birth to many extremely unimpressive children, that misconce

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1250,
    chunk_overlap=250,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_text(raw_text)

print(f"Book divided in {len(chunks)} chunks.")
print("\n--- Chunk 51 and 52 ---")
print(chunks[51])
print("")
print(chunks[52])

Book divided in 317 chunks.

--- Chunk 50 and 51 ---
bowed to me.
Monitor. I’d completely forgotten. She did mention something like that, I
now recalled. I also remembered how overwhelming the nausea was and ran
to the bathroom to throw up.
When I emerged from the bathroom with a completely empty stomach,
Miyagi was standing right on the other side of the doorway. It might have
been her job, but she didn’t know how to keep her distance. I pushed her out
of my way to get to the sink, where I washed my face and gargled, slugged
down a cup of water, then returned to my bed on the floor. My head was
killing me. The humidity wasn’t helping.
“As I explained to you yesterday,” said Miyagi, who was standing next to
my pillow now, “you have less than one year of life left, so from now on, you
will have a monitor at all times. Furthermore…”
“Can you go over this later?” I asked, outwardly annoyed. “As you can
see, I’m not in a great condition to listen now.”
“Very well. I’ll wait.”
Then Miyagi t

In [60]:
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse
import time

In [ ]:
QDRANT_URL=""
GEMINI_API_KEY=""

In [64]:
client = QdrantClient(url=QDRANT_URL)
collection_name = "hybrid_three_days_of_happiness"

In [65]:
dense_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)


client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)

print(f"Collection '{collection_name}' created with Hybrid support (Dense + Sparse).")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    sparse_vector_name="sparse"
)

Collection 'hybrid_three_days_of_happiness' created with Hybrid support (Dense + Sparse).


In [66]:
docs = []
for idx, chunk in enumerate(chunks): # Iteramos sobre la lista que generó LangChain
    
    metadata = {
        "source": "Three Days of Happiness",
        "chunk_index": idx
    }
    
    docs.append(Document(page_content=chunk, metadata=metadata))

batch_size = 80
total_batches = (len(docs) // batch_size) + 1

print(f"Initializing {total_batches} batches")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    batch_n = (i // batch_size) + 1
    
    print(f"Indexing {batch_n}/{total_batches} ({len(batch)} chunks)...")
    vector_store.add_documents(batch)
    
    if i + batch_size < len(docs):
        print("Limit reached, waiting 60 seconds to refresh Gemini free quota...")
        time.sleep(60)

print(f"Done, check {QDRANT_URL}/dashboard")

Initializing 4 batches
Indexing 1/4 (80 chunks)...
Limit reached, waiting 60 seconds to refresh Gemini free quota...
Indexing 2/4 (80 chunks)...
Limit reached, waiting 60 seconds to refresh Gemini free quota...
Indexing 3/4 (80 chunks)...
Limit reached, waiting 60 seconds to refresh Gemini free quota...
Indexing 4/4 (77 chunks)...
Done, check http://localhost:6333/dashboard
